# Mi perfil conjunto de calendar y calendar_dates

Consulté ambos snapshots mediante Genie el 10 de noviembre de 2026. Esta guía conserva siete consultas definitivas ejecutadas y sus explicaciones; sus resultados agregados están en [mi informe](../../../docs/calendars_profile_2025_2026.md). No guardo filas raw ni outputs en Git.

Todas las celdas son SQL de lectura y están fijadas a los dos pares versión/hash del proyecto. No instalan librerías, no crean tablas y no forman parte del Job. El contenedor del notebook todavía no tiene una ejecución completa documentada en Databricks; las consultas individuales sí se ejecutaron y recuperaron sin truncamiento.

Mi punto de atención es temporal: la fecha del ZIP no reemplaza las fechas de servicio publicadas. Después del perfil implementé los calendarios en el pipeline local. Su ejecución y publicación en Databricks siguen pendientes.

## Bloque 1 — Esquema actual de Bronze

Compruebo los tipos actuales de ambas tablas. Los 21 campos incluyen negocio y procedencia; los campos de negocio siguen como string y bronze_loaded_at_utc es timestamp. El esquema Delta no prueba qué encabezado publicó cada archivo.

In [ ]:
%sql
-- BLOQUE 1 — Lectura verificada mediante Genie; sin escrituras.
SELECT 
  table_name,
  column_name,
  ordinal_position,
  data_type,
  is_nullable
FROM system.information_schema.columns
WHERE table_catalog = 'workspace'
  AND table_schema = 'gtfs_bogota'
  AND table_name IN ('bronze_gtfs_calendar', 'bronze_gtfs_calendar_dates')
ORDER BY table_name, ordinal_position;


## Bloque 2 — Columnas publicadas por snapshot

Consulto la auditoría del encabezado por versión y hash. Calendar publicó diez campos y calendar_dates tres, iguales entre los dos snapshots. Estos nombres pueden estar ordenados por la auditoría; no afirmo conservar el orden físico del TXT.

In [ ]:
%sql
-- BLOQUE 2 — Lectura verificada mediante Genie; sin escrituras.
SELECT 
  source_feed_version,
  source_sha256,
  source_file_name,
  source_columns_json,
  observed_at_utc
FROM `workspace`.`gtfs_bogota`.`audit_gtfs_source_schema_observations`
WHERE source_file_name IN ('calendar.txt', 'calendar_dates.txt')
  AND ((source_feed_version = '2025-10-28' AND source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
    OR (source_feed_version = '2026-09-29' AND source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097'))
ORDER BY source_feed_version, source_file_name;


## Bloque 3 — Patrones semanales y sus intervalos

Cada fila es un patrón de fechas, no una agencia. En 2025 los siete intervalos van de 2000-01-01 a 2099-01-01; en 2026, de 2026-01-01 a 2026-12-31. No comparo IDs aislados entre versiones ni deduplico servicios porque compartan días semanales.

In [ ]:
%sql
-- BLOQUE 3 — Lectura verificada mediante Genie; sin escrituras.
SELECT 
  `service_id`,
  `monday`,
  `tuesday`,
  `wednesday`,
  `thursday`,
  `friday`,
  `saturday`,
  `sunday`,
  `start_date`,
  `end_date`,
  `source_feed_version`,
  `source_sha256`
FROM `workspace`.`gtfs_bogota`.`bronze_gtfs_calendar`
WHERE (source_feed_version = '2025-10-28' AND source_sha256 = '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952')
   OR (source_feed_version = '2026-09-29' AND source_sha256 = '08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')
ORDER BY source_feed_version, service_id
LIMIT 100;


## Bloque 4 — Completitud y espacios por campo

Esta agregación distingue nulo, cadena vacía, solo espacios Unicode y espacios extremos. Los 26 perfiles campo/snapshot no encontraron esos casos ni controles Unicode. Los conteos pertenecen a todo el snapshot seleccionado.

In [ ]:
%sql
-- BLOQUE 4 — Lectura verificada mediante Genie; sin escrituras.
with cells as (select 'calendar' as entity, source_feed_version, source_sha256, stack(10, 'service_id', `service_id`, 'monday', `monday`, 'tuesday', `tuesday`, 'wednesday', `wednesday`, 'thursday', `thursday`, 'friday', `friday`, 'saturday', `saturday`, 'sunday', `sunday`, 'start_date', `start_date`, 'end_date', `end_date`) as (field_name, raw_value) from workspace.gtfs_bogota.bronze_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) union all select 'calendar_dates' as entity, source_feed_version, source_sha256, stack(3, 'service_id', `service_id`, 'date', `date`, 'exception_type', `exception_type`) as (field_name, raw_value) from workspace.gtfs_bogota.bronze_gtfs_calendar_dates where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')))
select entity, source_feed_version, source_sha256, field_name, count(*) as snapshot_rows,
sum(case when raw_value is null then 1 else 0 end) as null_count,
sum(case when raw_value='' then 1 else 0 end) as empty_count,
sum(case when raw_value<>'' and raw_value rlike r'(?U)^\s*$' then 1 else 0 end) as whitespace_only_count,
count(distinct case when raw_value is not null and not (raw_value rlike r'(?U)^\s*$') then raw_value end) as distinct_non_blank_count,
sum(case when raw_value is not null and not (raw_value rlike r'(?U)^\s*$') and raw_value rlike r'(?U)^\s|\s$' then 1 else 0 end) as edge_whitespace_count,
sum(case when raw_value rlike r'[\p{Cc}]' then 1 else 0 end) as control_character_count
from cells group by entity, source_feed_version, source_sha256, field_name
order by entity, source_feed_version, field_name;


## Bloque 5 — Claves, copias y procedencia

Uso JSON de los campos de clave para evitar colisiones por concatenar con separadores. Mido copias fuente idénticas y claves con variantes; selecciono por hash para hacer visibles versiones o rutas incoherentes. Encontré 7 / 7 patrones y 140 / 133 excepciones, sin copias, conflictos ni discrepancias de procedencia.

In [ ]:
%sql
-- BLOQUE 5 — Lectura verificada mediante Genie; sin escrituras.
with raw as (select 'calendar' as entity, case source_sha256 when '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' then '2025-10-28' else '2026-09-29' end as expected_version, source_feed_version, source_sha256, source_file_path, bronze_loaded_at_utc, to_json(named_struct('service_id',`service_id`),map('ignoreNullFields','false')) as key_json, to_json(named_struct('service_id',`service_id`,'monday',`monday`,'tuesday',`tuesday`,'wednesday',`wednesday`,'thursday',`thursday`,'friday',`friday`,'saturday',`saturday`,'sunday',`sunday`,'start_date',`start_date`,'end_date',`end_date`),map('ignoreNullFields','false')) as business_json from workspace.gtfs_bogota.bronze_gtfs_calendar where source_sha256 in ('07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097') union all select 'calendar_dates' as entity, case source_sha256 when '07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952' then '2025-10-28' else '2026-09-29' end as expected_version, source_feed_version, source_sha256, source_file_path, bronze_loaded_at_utc, to_json(named_struct('service_id',`service_id`,'date',`date`),map('ignoreNullFields','false')) as key_json, to_json(named_struct('service_id',`service_id`,'date',`date`,'exception_type',`exception_type`),map('ignoreNullFields','false')) as business_json from workspace.gtfs_bogota.bronze_gtfs_calendar_dates where source_sha256 in ('07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952','08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')),
keys as (select entity,expected_version,source_sha256,key_json,count(*) as copies,count(distinct business_json) as variants from raw group by entity,expected_version,source_sha256,key_json),
counts as (select entity,expected_version,source_sha256,sum(copies) as input_rows,count(*) as distinct_raw_keys,sum(copies-variants) as identical_extra_copies,sum(case when variants>1 then 1 else 0 end) as conflicting_keys from keys group by entity,expected_version,source_sha256),
metadata as (select entity,expected_version,source_sha256,sum(case when not (source_feed_version <=> expected_version) then 1 else 0 end) as wrong_version_rows,
sum(case when not (source_file_path <=> concat('/Volumes/workspace/gtfs_bogota/staging_gtfs/gtfs_static/feed_version=',expected_version,'/sha256=',source_sha256,'/',entity,'.txt')) then 1 else 0 end) as wrong_path_rows,
sum(case when bronze_loaded_at_utc is null then 1 else 0 end) as null_timestamp_rows
from raw group by entity,expected_version,source_sha256)
select c.*,m.wrong_version_rows,m.wrong_path_rows,m.null_timestamp_rows from counts c join metadata m using(entity,expected_version,source_sha256) order by entity,expected_version;


## Bloque 6 — Excepciones, padres y contexto temporal

Agrupo calendar por clave antes de unir para evitar multiplicar excepciones. Los 14 grupos no encontraron IDs sin patrón, patrones ambiguos ni fechas fuera del intervalo semanal publicado. Las 140 excepciones del snapshot 2025 son de 2023; las 133 del snapshot 2026 son de 2026. Una diferencia de año se muestra como contexto, no como corrección automática.

In [ ]:
%sql
-- BLOQUE 6 — Lectura verificada mediante Genie; sin escrituras.
with c as (
select source_feed_version,source_sha256,service_id,count(distinct to_json(named_struct('service_id',`service_id`,'monday',`monday`,'tuesday',`tuesday`,'wednesday',`wednesday`,'thursday',`thursday`,'friday',`friday`,'saturday',`saturday`,'sunday',`sunday`,'start_date',`start_date`,'end_date',`end_date`),map('ignoreNullFields','false'))) as variants,min(try_cast(concat(substr(`start_date`,1,4),'-',substr(`start_date`,5,2),'-',substr(`start_date`,7,2)) as date)) as first_day,max(try_cast(concat(substr(`end_date`,1,4),'-',substr(`end_date`,5,2),'-',substr(`end_date`,7,2)) as date)) as last_day
from workspace.gtfs_bogota.bronze_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) group by source_feed_version,source_sha256,service_id),
d as (select source_feed_version,source_sha256,service_id,exception_type,try_cast(concat(substr(`date`,1,4),'-',substr(`date`,5,2),'-',substr(`date`,7,2)) as date) as service_date
from workspace.gtfs_bogota.bronze_gtfs_calendar_dates where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')))
select d.source_feed_version,d.source_sha256,d.service_id,d.exception_type,count(*) as rows,
min(d.service_date) as first_exception,max(d.service_date) as last_exception,
sum(case when c.service_id is null then 1 else 0 end) as missing_calendar_id_rows,
sum(case when c.variants>1 then 1 else 0 end) as ambiguous_calendar_rows,
sum(case when c.variants=1 and (d.service_date<c.first_day or d.service_date>c.last_day) then 1 else 0 end) as outside_calendar_interval_rows,
sum(case when year(d.service_date)<>year(try_cast(d.source_feed_version as date)) then 1 else 0 end) as different_snapshot_year_rows
from d left join c on d.source_feed_version=c.source_feed_version and d.source_sha256=c.source_sha256 and d.service_id=c.service_id
group by d.source_feed_version,d.source_sha256,d.service_id,d.exception_type
order by d.source_feed_version,d.service_id,d.exception_type;


## Bloque 7 — Fechas y dominios con conversión segura

Compruebo valores requeridos y dominios incluyendo null. Las fechas deben tener ocho dígitos y convertirse con try_cast sobre formato ISO: una fecha imposible queda detectable sin lanzar una excepción del parser. Los controles devolvieron cero valores ausentes, fechas inválidas y códigos fuera del dominio observado.

In [ ]:
%sql
-- BLOQUE 7 — Lectura verificada mediante Genie; sin escrituras.
with cells as (select 'calendar' as entity, source_feed_version, source_sha256, stack(10, 'service_id', `service_id`, 'monday', `monday`, 'tuesday', `tuesday`, 'wednesday', `wednesday`, 'thursday', `thursday`, 'friday', `friday`, 'saturday', `saturday`, 'sunday', `sunday`, 'start_date', `start_date`, 'end_date', `end_date`) as (field_name, raw_value) from workspace.gtfs_bogota.bronze_gtfs_calendar where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')) union all select 'calendar_dates' as entity, source_feed_version, source_sha256, stack(3, 'service_id', `service_id`, 'date', `date`, 'exception_type', `exception_type`) as (field_name, raw_value) from workspace.gtfs_bogota.bronze_gtfs_calendar_dates where ((source_feed_version='2025-10-28' and source_sha256='07ba2ba9bb3b5bc1e38357355956757ce84fc68e6d05b74577bf017e189ab952') or (source_feed_version='2026-09-29' and source_sha256='08d14e0b3a5b9caa5451b722659871fd554e3273b94bdd770fdf81a26bf02097')))
select entity,source_feed_version,source_sha256,
sum(case when raw_value is null or raw_value rlike r'(?U)^\s*$' then 1 else 0 end) as missing_required_values,
sum(case when field_name in ('start_date','end_date','date') and
(raw_value is null or not (raw_value rlike r'^[0-9]{8}$') or
 try_cast(concat(substr(raw_value,1,4),'-',substr(raw_value,5,2),'-',substr(raw_value,7,2)) as date) is null) then 1 else 0 end) as invalid_date_values,
sum(case when field_name in ('monday','tuesday','wednesday','thursday','friday','saturday','sunday')
and (raw_value is null or raw_value not in ('0','1')) then 1
when field_name='exception_type' and (raw_value is null or raw_value not in ('1','2')) then 1 else 0 end) as invalid_enum_values
from cells group by entity,source_feed_version,source_sha256 order by entity,source_feed_version;


## Qué queda por decidir

Implementé IDs string, fechas date y códigos int, conservando los valores fuente. Mantendré la identidad por versión + hash + clave, las fechas originales de 2023 y los intervalos publicados. El [contrato implementado](../../../docs/contracts/calendars.md) distingue errores de formato de advertencias de alcance temporal.

Antes de derivar service_dates cerraré el horizonte de fechas y cómo se mostrará la limitación de 2025 en Gold/Power BI. No expandiré aquí 99 años ni trasladaré festivos de 2023 a otro año.